# Ordinal Validity Evaluation of GEV ξ Parameter

This evaluation assesses whether single-sentence GEV shape parameter (ξ) functions as a valid ordinal index of tail-constraint severity despite failing super-block parametric validation.

**Six complementary analyses:**
1. **Super-block framing** — Extract metrics showing why parametric super-block validation fails (degenerate fits, negative correlations)
2. **Bootstrap ranking stability** — Resample ξ from N(ξ_raw, SE) and measure rank consistency via Kendall τ
3. **Rank-based regression** — Test ξ_rank ~ entropy_rank + morph_rank + hdr_rank
4. **Permutation test** — Distribution-free test for ξ–entropy association
5. **Simulation validation** — Generate synthetic GEV data, recover ξ via L-moments, check rank recovery
6. **Overall verdict** — Combine four tests into a single ordinal validity verdict

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# statsmodels — NOT pre-installed on Colab at needed version
# (actually pre-installed, but putting in guard to be safe)

# Core packages: pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scipy==1.16.3',
         'matplotlib==3.10.0', 'statsmodels==0.14.6')

In [ ]:
import json
import math
import os

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import brentq
from scipy.special import gamma as gamma_fn
import statsmodels.api as sm
import matplotlib.pyplot as plt

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-42dac1-word-order-entropy-predicts-ordinal-tail/main/evaluation_iter3_ordinal_validit/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded {len(data['treebanks'])} treebanks, "
      f"{len(data['simulation_results'])} simulation results")
print(f"Reference verdict: {data['metadata']['verdict']}")

In [ ]:
# ── Configuration: tunable parameters ──────────────────────────
# Start with MINIMUM values for fast demo; original values commented.
N_BOOTSTRAP = 10         # original: 500
N_PERMUTATIONS = 200     # original: 10_000
N_SYNTHETIC = 10         # original: 100
SAMPLE_SIZES = [50, 200] # original: [50, 100, 200, 500]
XI_RANGE = (-1.5, -0.1)

## Prepare Treebank DataFrame

Convert the loaded treebank data into a pandas DataFrame for analysis.

In [ ]:
df = pd.DataFrame(data["treebanks"])
print(f"Treebank DataFrame: {len(df)} rows")
print(df[["treebank_id", "xi_raw", "word_order_entropy", "family"]].head(10).to_string(index=False))

## Step 1 — Super-block Framing Metrics

Extract pre-computed super-block framing metrics. These explain why parametric super-block validation fails: negative correlations arise from ceiling concentration in bounded integer data, with 40.7% degenerate fits.

In [ ]:
# Step 1: Super-block framing (pre-computed from exp_id3)
sb_m = data["superblock_metrics"]
print("Step 1: Super-block Framing Metrics")
print(f"  K=20 raw Spearman rho: {sb_m['k20_raw_rho']:.4f}")
print(f"  K=30 raw Spearman rho: {sb_m['k30_raw_rho']:.4f}")
print(f"  Quantile p95 rho:      {sb_m['quantile_p95_rho']:.4f}")
print(f"  Quantile p99 rho:      {sb_m['quantile_p99_rho']:.4f}")
print(f"  Degenerate fits:       {sb_m['pct_degenerate_super_fits']:.1f}%")

## Step 2 — Bootstrap Ranking Stability

Resample ξ values from N(ξ_raw, SE) for each treebank across `N_BOOTSTRAP` iterations, then measure pairwise Kendall τ between bootstrap rankings. Threshold: mean τ > 0.80 to pass.

In [ ]:
# Step 2: Bootstrap ranking stability
n_tb = len(df)
xi = df["xi_raw"].values.copy()
se = np.maximum(df["xi_raw_se"].values.copy(), 1e-6)

# Generate bootstrap rankings
ranks_list = []
for i in range(N_BOOTSTRAP):
    rng = np.random.default_rng(i)
    ranks_list.append(stats.rankdata(rng.normal(xi, se)))
ranks = np.array(ranks_list)  # (N_BOOTSTRAP, n_tb)

# Pairwise Kendall tau and Spearman rho
rng = np.random.default_rng(42)
n_pairs = min(500, N_BOOTSTRAP * (N_BOOTSTRAP - 1) // 2)
taus, rhos = [], []
for _ in range(n_pairs):
    i, j = rng.choice(N_BOOTSTRAP, 2, replace=False)
    t, _ = stats.kendalltau(ranks[i], ranks[j])
    r, _ = stats.spearmanr(ranks[i], ranks[j])
    if not np.isnan(t): taus.append(float(t))
    if not np.isnan(r): rhos.append(float(r))

tau_arr = np.array(taus)
mean_tau = float(np.mean(tau_arr))
sd_tau = float(np.std(tau_arr))
ci_lo = float(np.percentile(tau_arr, 2.5))
ci_hi = float(np.percentile(tau_arr, 97.5))
mean_rho = float(np.mean(rhos))
stable = mean_tau > 0.80

boot_m = {
    "mean_kendall_tau": mean_tau,
    "sd_kendall_tau": sd_tau,
    "ci95_kendall_tau_lo": ci_lo,
    "ci95_kendall_tau_hi": ci_hi,
    "mean_spearman_rho": mean_rho,
    "ranking_stable": 1 if stable else 0,
    "n_treebanks_used": n_tb,
    "n_bootstrap_iterations": N_BOOTSTRAP,
}

# Per-treebank bootstrap rank statistics
boot_tb = {}
for idx in range(n_tb):
    r = ranks[:, idx]
    boot_tb[df.iloc[idx]["treebank_id"]] = {
        "mean_rank": float(np.mean(r)),
        "sd_rank": float(np.std(r)),
        "ci_lo": float(np.percentile(r, 2.5)),
        "ci_hi": float(np.percentile(r, 97.5)),
    }

print(f"Step 2: Bootstrap Ranking Stability ({N_BOOTSTRAP} iterations, {n_tb} treebanks)")
print(f"  Mean Kendall tau: {mean_tau:.4f} (SD={sd_tau:.4f})")
print(f"  95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")
print(f"  Mean Spearman rho: {mean_rho:.4f}")
print(f"  Stable (tau > 0.80): {'PASS' if stable else 'FAIL'}")

## Step 3 — Rank-based Regression

Fit OLS: ξ_rank ~ entropy_rank + morph_rank + hdr_rank. The test checks whether entropy is significant at rank level and whether morphological richness and head direction ratio are non-significant (matching parametric results).

In [ ]:
# Step 3: Rank-based regression
xi_rank = stats.rankdata(df["xi_raw"].values)
ent_rank = stats.rankdata(df["word_order_entropy"].values)
morph_rank = stats.rankdata(df["morph_richness"].values)
hdr_rank = stats.rankdata(df["head_direction_ratio"].values)

# Bivariate Spearman: xi <-> entropy
rho_ent, p_ent = stats.spearmanr(df["xi_raw"].values,
                                  df["word_order_entropy"].values)

# OLS rank regression
X = sm.add_constant(np.column_stack([ent_rank, morph_rank, hdr_rank]))
model = sm.OLS(xi_rank, X).fit()

reg_m = {
    "spearman_rho_xi_entropy": float(rho_ent),
    "spearman_p_xi_entropy": float(p_ent),
    "rank_reg_beta_entropy": float(model.params[1]),
    "rank_reg_p_entropy": float(model.pvalues[1]),
    "rank_reg_beta_morph": float(model.params[2]),
    "rank_reg_p_morph": float(model.pvalues[2]),
    "rank_reg_beta_hdr": float(model.params[3]),
    "rank_reg_p_hdr": float(model.pvalues[3]),
    "rank_reg_r_squared": float(model.rsquared),
}

# Parametric agreement: entropy significant, morph/hdr not
ent_sig = reg_m["rank_reg_p_entropy"] < 0.05
morph_ns = reg_m["rank_reg_p_morph"] >= 0.05
hdr_ns = reg_m["rank_reg_p_hdr"] >= 0.05
agree = ent_sig and morph_ns and hdr_ns
reg_m["parametric_agreement"] = 1 if agree else 0

print("Step 3: Rank-based Regression")
print(f"  Spearman rho (xi, entropy): {rho_ent:.4f} (p={p_ent:.6f})")
print(f"  R-squared: {reg_m['rank_reg_r_squared']:.4f}")
print(f"  beta_entropy={reg_m['rank_reg_beta_entropy']:.4f} (p={reg_m['rank_reg_p_entropy']:.6f})")
print(f"  beta_morph  ={reg_m['rank_reg_beta_morph']:.4f} (p={reg_m['rank_reg_p_morph']:.6f})")
print(f"  beta_hdr    ={reg_m['rank_reg_beta_hdr']:.4f} (p={reg_m['rank_reg_p_hdr']:.6f})")
print(f"  Parametric agreement: {'PASS' if agree else 'FAIL'}")

## Step 4 — Permutation Test

Distribution-free permutation test: shuffle entropy values `N_PERMUTATIONS` times, compute Spearman ρ each time, and check if the observed ρ is extreme enough (p < 0.001 threshold).

In [ ]:
# Step 4: Permutation test
xi_vals = df["xi_raw"].values
ent_vals = df["word_order_entropy"].values
obs_rho, _ = stats.spearmanr(xi_vals, ent_vals)

rng = np.random.default_rng(42)
perm_rhos = np.empty(N_PERMUTATIONS)
for i in range(N_PERMUTATIONS):
    perm_rhos[i] = stats.spearmanr(xi_vals, rng.permutation(ent_vals))[0]

n_exceed = int(np.sum(np.abs(perm_rhos) >= np.abs(obs_rho)))
p_val = n_exceed / N_PERMUTATIONS

perm_m = {
    "observed_rho": float(obs_rho),
    "permutation_p_value": float(p_val),
    "n_permutations": N_PERMUTATIONS,
    "n_exceeding": n_exceed,
    "permutation_ci95_lo": float(np.percentile(perm_rhos, 2.5)),
    "permutation_ci95_hi": float(np.percentile(perm_rhos, 97.5)),
}

print(f"Step 4: Permutation Test ({N_PERMUTATIONS} iterations)")
print(f"  Observed rho: {obs_rho:.4f}")
print(f"  Permutation p-value: {p_val:.6f} ({n_exceed}/{N_PERMUTATIONS})")
print(f"  Null 95% CI: [{perm_m['permutation_ci95_lo']:.4f}, {perm_m['permutation_ci95_hi']:.4f}]")
print(f"  Significant (p < 0.001): {'PASS' if p_val < 0.001 else 'FAIL'}")

## Step 5 — Simulation Validation (L-moments GEV Fitting)

Generate synthetic GEV data with known ξ values, recover ξ via L-moments fitting, and check rank recovery (Spearman ρ > 0.85 threshold). This validates the L-moments estimation method itself.

In [ ]:
# L-moments GEV fitting functions (from eval.py)

def sample_lmoments(x):
    """First three sample L-moments via unbiased PWM estimators."""
    n = len(x)
    if n < 3:
        return float(np.mean(x)), 0.0, 0.0
    xs = np.sort(x).astype(np.float64)
    i = np.arange(1, n + 1, dtype=np.float64)
    b0 = np.mean(xs)
    b1 = np.dot((i - 1) / (n * (n - 1)), xs)
    b2 = np.dot((i - 1) * (i - 2) / (n * (n - 1) * (n - 2)), xs)
    l1 = b0
    l2 = 2.0 * b1 - b0
    l3 = 6.0 * b2 - 6.0 * b1 + b0
    t3 = l3 / l2 if abs(l2) > 1e-15 else 0.0
    return float(l1), float(l2), float(t3)


def gev_lmom_fit(x):
    """Fit GEV via L-moments. Returns (loc, scale, k)."""
    l1, l2, t3 = sample_lmoments(x)
    if abs(l2) < 1e-15:
        return l1, 0.0, 0.0
    GUMBEL_TAU3 = 2.0 * math.log(3) / math.log(2) - 3.0

    def tau3_eq(k):
        if abs(k) < 1e-10:
            return t3 - GUMBEL_TAU3
        return t3 - (2.0 * (1.0 - 3.0 ** (-k)) / (1.0 - 2.0 ** (-k)) - 3.0)

    try:
        k = brentq(tau3_eq, -2.0, 10.0, xtol=1e-12)
    except ValueError:
        k = 0.0

    if abs(k) < 1e-10:
        sigma = l2 / math.log(2)
        mu = l1 - sigma * 0.5772156649
    else:
        try:
            g1 = float(gamma_fn(1.0 + k))
        except (ValueError, OverflowError):
            return l1, abs(l2), 0.0
        denom = (1.0 - 2.0 ** (-k)) * g1
        if abs(denom) < 1e-15:
            return l1, abs(l2), 0.0
        sigma = l2 * k / denom
        mu = l1 - sigma * (1.0 - g1) / k

    return float(mu), float(max(sigma, 1e-15)), float(k)


# Step 5: Simulation validation
print(f"Step 5: Simulation Validation ({N_SYNTHETIC} synthetic x {len(SAMPLE_SIZES)} sizes)")
true_xis = np.linspace(XI_RANGE[0], XI_RANGE[1], N_SYNTHETIC)

results = []
seed = 0
for n in SAMPLE_SIZES:
    for true_xi in true_xis:
        rng = np.random.default_rng(seed)
        c_scipy = -true_xi  # sign convention
        sim_data = stats.genextreme.rvs(c=c_scipy, loc=5.0, scale=2.0,
                                        size=n, random_state=rng)
        try:
            _, _, k_est = gev_lmom_fit(sim_data)
            xi_est = -k_est
        except Exception:
            xi_est = float("nan")
        results.append((float(true_xi), xi_est, n))
        seed += 1

# Group by sample size
by_n = {}
for t_xi, e_xi, n in results:
    by_n.setdefault(n, []).append((t_xi, e_xi))

# Overall (valid only)
valid = [(t, e) for t, e, _ in results if not np.isnan(e)]
all_true = np.array([t for t, _ in valid])
all_est = np.array([e for _, e in valid])
overall_rho, _ = stats.spearmanr(all_true, all_est)

# Per-n recovery
rec_by_n = {}
for n in sorted(by_n):
    pairs = [(t, e) for t, e in by_n[n] if not np.isnan(e)]
    if len(pairs) >= 3:
        tarr, earr = zip(*pairs)
        r, _ = stats.spearmanr(tarr, earr)
        rec_by_n[n] = float(r)

errors = all_est - all_true
passes = overall_rho > 0.85

sim_m = {
    "rank_recovery_rho": float(overall_rho),
    "rank_recovery_passes": 1 if passes else 0,
    "xi_rmse": float(np.sqrt(np.mean(errors ** 2))),
    "xi_bias": float(np.mean(errors)),
    "n_synthetic_treebanks": float(N_SYNTHETIC),
    "true_xi_range_lo": float(XI_RANGE[0]),
    "true_xi_range_hi": float(XI_RANGE[1]),
}
for n, r in rec_by_n.items():
    sim_m[f"rank_recovery_n{n}"] = r

# Collect simulation examples for visualization
sim_ex = []
ref_n = SAMPLE_SIZES[-1]
if ref_n in by_n:
    for t_xi, e_xi in by_n[ref_n]:
        if not np.isnan(e_xi):
            sim_ex.append({"true_xi": t_xi, "est_xi": e_xi,
                           "error": e_xi - t_xi, "n": ref_n})

print(f"  Overall rank recovery rho: {overall_rho:.4f}")
print(f"  Passes (rho > 0.85): {'PASS' if passes else 'FAIL'}")
print(f"  RMSE: {sim_m['xi_rmse']:.4f}, Bias: {sim_m['xi_bias']:.4f}")
for n, r in sorted(rec_by_n.items()):
    print(f"    n={n}: rho={r:.4f}")

## Step 6 — Overall Verdict

Combine the four tests into a single ordinal validity verdict:
- Bootstrap stable (τ > 0.80)
- Rank regression agrees (entropy significant, morph/hdr not)
- Permutation significant (p < 0.001)
- Simulation recovers (ρ > 0.85)

In [ ]:
# Step 6: Overall verdict
bs = boot_m["ranking_stable"] == 1
rr = reg_m["parametric_agreement"] == 1
ps = perm_m["permutation_p_value"] < 0.001
sr = sim_m["rank_recovery_passes"] == 1

count = sum([bs, rr, ps, sr])
if count == 4:
    verdict = "VALIDATED"
elif count == 3:
    verdict = "PARTIALLY_VALIDATED"
elif count == 2:
    verdict = "WEAK"
else:
    verdict = "FAILED"

verd_m = {
    "bootstrap_stable": int(bs),
    "rank_regression_agrees": int(rr),
    "permutation_significant": int(ps),
    "simulation_recovers": int(sr),
    "all_four_pass": int(count == 4),
    "ordinal_validity_verdict_code": float(count),
}

print(f"{'='*50}")
print(f"VERDICT: {verdict} ({count}/4 pass)")
print(f"{'='*50}")
print(f"  Bootstrap stable:        {'PASS' if bs else 'FAIL'} (tau={boot_m['mean_kendall_tau']:.4f})")
print(f"  Rank regression agrees:  {'PASS' if rr else 'FAIL'}")
print(f"  Permutation significant: {'PASS' if ps else 'FAIL'} (p={perm_m['permutation_p_value']:.6f})")
print(f"  Simulation recovers:     {'PASS' if sr else 'FAIL'} (rho={sim_m['rank_recovery_rho']:.4f})")

## Visualization

Four panels summarizing the ordinal validity evaluation:
1. **Bootstrap rank stability** — distribution of pairwise Kendall τ
2. **Permutation null distribution** — observed ρ vs null
3. **Simulation recovery** — true vs estimated ξ
4. **Verdict summary** — pass/fail for each test

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Panel 1: Bootstrap Kendall tau distribution
ax = axes[0, 0]
ax.hist(tau_arr, bins=20, color="steelblue", edgecolor="white", alpha=0.8)
ax.axvline(0.80, color="red", linestyle="--", linewidth=2, label="Threshold (0.80)")
ax.axvline(mean_tau, color="orange", linestyle="-", linewidth=2, label=f"Mean tau={mean_tau:.3f}")
ax.set_xlabel("Pairwise Kendall tau")
ax.set_ylabel("Frequency")
ax.set_title("Bootstrap Ranking Stability")
ax.legend(fontsize=9)

# Panel 2: Permutation null distribution
ax = axes[0, 1]
ax.hist(perm_rhos, bins=30, color="lightgray", edgecolor="white", alpha=0.8, label="Null distribution")
ax.axvline(obs_rho, color="red", linestyle="-", linewidth=2, label=f"Observed rho={obs_rho:.3f}")
ax.axvline(-obs_rho, color="red", linestyle="--", linewidth=1, alpha=0.5)
ax.set_xlabel("Spearman rho (permuted)")
ax.set_ylabel("Frequency")
ax.set_title(f"Permutation Test (p={p_val:.4f})")
ax.legend(fontsize=9)

# Panel 3: Simulation recovery (true vs estimated xi)
ax = axes[1, 0]
if sim_ex:
    true_vals = [s["true_xi"] for s in sim_ex]
    est_vals = [s["est_xi"] for s in sim_ex]
    ax.scatter(true_vals, est_vals, alpha=0.6, s=30, color="steelblue", edgecolor="white")
    lims = [min(true_vals + est_vals) - 0.1, max(true_vals + est_vals) + 0.1]
    ax.plot(lims, lims, "k--", linewidth=1, label="Perfect recovery")
    ax.set_xlim(lims)
    ax.set_ylim(lims)
ax.set_xlabel("True xi")
ax.set_ylabel("Estimated xi (L-moments)")
ax.set_title(f"Simulation Recovery (rho={overall_rho:.3f}, n={ref_n})")
ax.legend(fontsize=9)

# Panel 4: Verdict summary
ax = axes[1, 1]
ax.axis("off")
tests = [
    ("Bootstrap Stable", bs, f"tau={boot_m['mean_kendall_tau']:.3f}"),
    ("Rank Regression", rr, f"R2={reg_m['rank_reg_r_squared']:.3f}"),
    ("Permutation Test", ps, f"p={perm_m['permutation_p_value']:.4f}"),
    ("Simulation Recovery", sr, f"rho={sim_m['rank_recovery_rho']:.3f}"),
]
for i, (name, passed, detail) in enumerate(tests):
    color = "green" if passed else "red"
    symbol = "PASS" if passed else "FAIL"
    ax.text(0.1, 0.85 - i * 0.2, f"{symbol}  {name}", fontsize=14, fontweight="bold",
            color=color, transform=ax.transAxes, family="monospace")
    ax.text(0.65, 0.85 - i * 0.2, detail, fontsize=12,
            color="gray", transform=ax.transAxes)
ax.text(0.1, 0.05, f"VERDICT: {verdict} ({count}/4)",
        fontsize=16, fontweight="bold", transform=ax.transAxes,
        color="green" if count == 4 else ("orange" if count >= 2 else "red"))
ax.set_title("Ordinal Validity Verdict")

plt.tight_layout()
plt.savefig("ordinal_validity_results.png", dpi=100, bbox_inches="tight")
plt.show()
print("Figure saved to ordinal_validity_results.png")